In [ ]:
import pandas as pd
from datasets import load_dataset
from datasets import Dataset
df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [ ]:
df = df.drop(columns=["image"])
print("First 5 rows of the DataFrame:")
print(df.head())
print(df.shape)

First 5 rows of the DataFrame:
               caption                                           captions
0       Tomato healthy  [A vibrant green and healthy tomato leaf with ...
1   Tomato Late blight  [A tomato leaf showing dark brown lesions and ...
2       Tomato healthy  [A vibrant green and healthy tomato leaf with ...
3  Tomato mosaic virus  [A tomato leaf with mosaic-like patterns of li...
4  Pepper bell healthy  [A fresh green bell pepper leaf with a smooth,...
(20638, 2)


In [ ]:
caption = df["caption"].unique().tolist()
caption, len(caption)

(['Tomato healthy',
  'Tomato Late blight',
  'Tomato mosaic virus',
  'Pepper bell healthy',
  'Potato Early blight',
  'Tomato Early blight',
  'Tomato YellowLeaf Curl Virus',
  'Tomato Target Spot',
  'Pepper bell Bacterial spot',
  'Tomato Septoria leaf spot',
  'Tomato Spider mites Two spotted spider mite',
  'Tomato Bacterial spot',
  'Potato Late blight',
  'Tomato Leaf Mold',
  'Potato healthy'],
 15)

In [ ]:
df.to_csv("dataset.csv", index=False)
df  = pd.read_csv("dataset.csv")
df.head()

,caption,captions
0,Tomato healthy,['A vibrant green and healthy tomato leaf with...
1,Tomato Late blight,['A tomato leaf showing dark brown lesions and...
2,Tomato healthy,['A vibrant green and healthy tomato leaf with...
3,Tomato mosaic virus,['A tomato leaf with mosaic-like patterns of l...
4,Pepper bell healthy,['A fresh green bell pepper leaf with a smooth...


In [ ]:
from sklearn.preprocessing import LabelEncoder
num_labels = 15
data_col_name = "captions"
label_col_name = "caption"
encoder = LabelEncoder()
encoder.fit(df[label_col_name].tolist())
df["label"] = encoder.transform(df[label_col_name].tolist())

In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.8)

In [ ]:
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [ ]:
def tokenize_fn(data):
  return tokenizer(data["captions"], truncation=True)

# Convert pandas DataFrames to datasets.Dataset objects
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4127 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
import evaluate
from datasets import load_dataset

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=15)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
training_arguments = TrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=0.00005,
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=2,
    weight_decay=0.01,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-3340522237.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Step,Training Loss
516,0.259800
1032,0.002800


TrainOutput(global_step=1032, training_loss=0.13132686889910883, metrics={'train_runtime': 108.7967, 'train_samples_per_second': 75.866, 'train_steps_per_second': 9.486, 'total_flos': 229778894232810.0, 'train_loss': 0.13132686889910883, 'epoch': 2.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
trainer.save_model("./best_model")
model_path="./best_model"

In [ ]:
predictions = trainer.predict(tokenized_test)
predicted_labels = encoder.inverse_transform(np.argmax(predictions.predictions, axis=-1))

print("Predicted Labels | True Labels")
print("----------------|------------")
for i in range(10):
  print(f"{predicted_labels[i]:<15} | {test_df['caption'].iloc[i]}")

Predicted Labels | True Labels
----------------|------------
Potato Late blight | Potato Late blight
Tomato Target Spot | Tomato Target Spot
Tomato Spider mites Two spotted spider mite | Tomato Spider mites Two spotted spider mite
Tomato healthy  | Tomato healthy
Tomato Early blight | Tomato Early blight
Tomato YellowLeaf Curl Virus | Tomato YellowLeaf Curl Virus
Tomato Spider mites Two spotted spider mite | Tomato Spider mites Two spotted spider mite
Tomato Spider mites Two spotted spider mite | Tomato Spider mites Two spotted spider mite
Tomato healthy  | Tomato healthy
Tomato Late blight | Tomato Late blight


In [ ]:
from google.colab import files
# Zip the directory before downloading
!zip -r best_model.zip best_model/
files.download('best_model.zip')

  adding: best_model/ (stored 0%)
  adding: best_model/vocab.txt (deflated 53%)
  adding: best_model/config.json (deflated 60%)
  adding: best_model/training_args.bin (deflated 53%)
  adding: best_model/tokenizer.json (deflated 71%)
  adding: best_model/tokenizer_config.json (deflated 75%)
  adding: best_model/special_tokens_map.json (deflated 42%)
  adding: best_model/model.safetensors (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>